In [1]:
!pip install pandas


In [2]:
import pandas as pd

df = pd.read_csv("../Assignment/contract_evaluation_dataset_mayank_charde.csv")
df.head()

,contract_text,expected_apr,expected_term,expected_payment,expected_penalty
0,The loan agreement offers an APR of 8.5% for a...,8.5,36.0,310.0,Late payment penalty of $40
1,This car loan has an APR of 7.9% with a repaym...,7.9,48.0,275.0,NaN
2,The borrower agrees to a loan term of 60 month...,NaN,60.0,350.0,Penalty of 3% on delayed payments
3,The agreement specifies a loan duration of 24 ...,NaN,24.0,NaN,NaN
4,"In case of late payment, a fee of $100 will be...",NaN,NaN,NaN,Late fee of $100


In [3]:

expected_output_schema = {
    "apr": None,
    "term_months": None,
    "monthly_payment": None,
    "penalty_clause": None
}

In [4]:
PROMPT_TEMPLATE = """
You are an information extraction system.

Extract ONLY the following fields from the contract text below:
- APR
- Term (in months)
- Monthly payment
- Penalty clause

Rules:
- Return output in valid JSON format only
- If a value is NOT explicitly mentioned, return null
- Do NOT infer, calculate, or assume any values
- Do NOT add extra fields
- Do NOT explain anything

Return JSON in this exact structure:
{{
  "apr": null,
  "term_months": null,
  "monthly_payment": null,
  "penalty_clause": null
}}

Contract text:
\"\"\"{contract_text}\"\"\"
"""

In [5]:

def extract_contract_fields(contract_text, llm_client):
    prompt = PROMPT_TEMPLATE.format(contract_text=contract_text)
    response = llm_client(prompt)
    return json.loads(response)

In [6]:

sample_df = df.sample(5, random_state=42)
sample_df

,contract_text,expected_apr,expected_term,expected_payment,expected_penalty
9,The car loan is offered at an APR of 6.8% for ...,6.8,36.0,NaN,NaN
25,The car loan is issued for a term of 90 months.,NaN,90.0,NaN,NaN
8,The borrower must repay the loan over 84 month...,NaN,84.0,NaN,Penalty of 5% on late payment
21,This contract specifies a loan term of 30 mont...,8.9,30.0,NaN,Late fee of $45
0,The loan agreement offers an APR of 8.5% for a...,8.5,36.0,310.0,Late payment penalty of $40


In [7]:

def dummy_llm_client(prompt):
    return """
    {
        "apr": null,
        "term_months": null,
        "monthly_payment": null,
        "penalty_clause": null
    }
    """

In [9]:
import json
results = []

for _, row in sample_df.iterrows():
    extracted = extract_contract_fields(
        row["contract_text"],
        dummy_llm_client
    )

    results.append({
        "contract_text": row["contract_text"],
        "llm_apr": extracted["apr"],
        "llm_term": extracted["term_months"],
        "llm_payment": extracted["monthly_payment"],
        "llm_penalty": extracted["penalty_clause"],
        "expected_apr": row["expected_apr"],
        "expected_term": row["expected_term"],
        "expected_payment": row["expected_payment"],
        "expected_penalty": row["expected_penalty"]
    })

results_df = pd.DataFrame(results)
results_df

,contract_text,llm_apr,llm_term,llm_payment,llm_penalty,expected_apr,expected_term,expected_payment,expected_penalty
0,The car loan is offered at an APR of 6.8% for ...,None,None,None,None,6.8,36.0,NaN,NaN
1,The car loan is issued for a term of 90 months.,None,None,None,None,NaN,90.0,NaN,NaN
2,The borrower must repay the loan over 84 month...,None,None,None,None,NaN,84.0,NaN,Penalty of 5% on late payment
3,This contract specifies a loan term of 30 mont...,None,None,None,None,8.9,30.0,NaN,Late fee of $45
4,The loan agreement offers an APR of 8.5% for a...,None,None,None,None,8.5,36.0,310.0,Late payment penalty of $40


In [10]:

def exact_match(llm_value, expected_value):
    return int(str(llm_value).strip() == str(expected_value).strip())

In [11]:
results_df["apr_match"] = results_df.apply(
    lambda r: exact_match(r["llm_apr"], r["expected_apr"]), axis=1
)

results_df["term_match"] = results_df.apply(
    lambda r: exact_match(r["llm_term"], r["expected_term"]), axis=1
)

results_df["payment_match"] = results_df.apply(
    lambda r: exact_match(r["llm_payment"], r["expected_payment"]), axis=1
)

results_df["penalty_match"] = results_df.apply(
    lambda r: exact_match(r["llm_penalty"], r["expected_penalty"]), axis=1
)

results_df

,contract_text,llm_apr,llm_term,llm_payment,llm_penalty,expected_apr,expected_term,expected_payment,expected_penalty,apr_match,term_match,payment_match,penalty_match
0,The car loan is offered at an APR of 6.8% for ...,None,None,None,None,6.8,36.0,NaN,NaN,0,0,0,0
1,The car loan is issued for a term of 90 months.,None,None,None,None,NaN,90.0,NaN,NaN,0,0,0,0
2,The borrower must repay the loan over 84 month...,None,None,None,None,NaN,84.0,NaN,Penalty of 5% on late payment,0,0,0,0
3,This contract specifies a loan term of 30 mont...,None,None,None,None,8.9,30.0,NaN,Late fee of $45,0,0,0,0
4,The loan agreement offers an APR of 8.5% for a...,None,None,None,None,8.5,36.0,310.0,Late payment penalty of $40,0,0,0,0
